In [1]:
import math
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F


In [2]:
!nvidia-smi

Mon Sep 21 10:57:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             62W /  400W |   13002MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
class LayerNorm(nn.Module):
    "Pytorch doesnt support option bias equals false "
    def __init__(self,ndim,bias):
        super().__init__()
        self.weight=nn.Parameter(torch.ones(ndim))
        self.bias=nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self,input):
        return F.layer_norm(input,self.weight.shape,self.weight,self.bias,1e-5)

In [4]:
class CausalSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_embd % config.n_head==0
        self.c_attn=nn.Linear(config.n_embd,3 * config.n_embd,bias=config.bias) #batched k,v ,q
        self.c_proj=nn.Linear(config.n_embd,config.n_embd,bias=config.bias) #output projection
        #regularization
        self.attn_dropot=nn.Dropout(config.dropout)
        self.resid_dropout=nn.Dropout(config.dropout)
        self.n_head=config.n_head
        self.n_embd=config.n_embd
        self.dropout=config.dropout

        self.flash= hasattr(torch.nn.functional,'scaled_dot_product_attention')
        if not self.flash:
            print('Using slow attention....')
            self.register_buffer('bias',torch.tril(torch.ones(config.block_size,config.block_size))
                                 .view(1,1,config.block_size,config.block_size))

    def forward(self,x):
        B,T,C=x.size() #batch size , sequence length , embedding dimnsion

        q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2) # (B, nh, T, hs)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2) # (B, nh, T, hs)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2) # (B, nh, T, hs)
        #(B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # print('using flash attention')
            y=torch.nn.functional.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            att=(q @ k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0,float('-inf')) #type:ignore
            att=F.softmax(att,dim=-1)
            att=self.attn_dropot(att)
            y=att @ v #(B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y=y.transpose(1,2).contiguous().view(B,T,C) #re assemble all heads side by side


        #output projection
        y=self.resid_dropout(self.c_proj(y))
        return y
    
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x
    
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x







In [5]:
@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # 50257 - GPT -2 vocab size, padded up to neares 64 multiple
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

In [ ]:
class GPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config=config

        self.transformer=nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size,config.n_embd),
            wpe=nn.Embedding(config.block_size,config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd,bias=config.bias)
        ))

        self.lm_head=nn.Linear(config.n_embd, config.vocab_size,bias=False)
        self.transformer.wte.weight= self.lm_head.weight #weight tying scheme

        self.apply(self._init_weights)

        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn,p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p,mean=0.0,std=0.02/math.sqrt(2 * config.n_layer)) # 2 comes from every single block having 2 ( attn,mlp) to add to the residual stream


        print(f"number of parameters : {self.get_num_params()/1e6} M")


    def get_num_params(self,non_embedding=True):
        """ returning total number of parameters for the model
        for non embedding equals true the pos embd parameters get subtracted ,
        not subtractting the wte as these are tied to final layer"""
        n_params= sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            torch.nn.init.normal_(module.weight,mean=0.0,std=0.02) #kinda similiar to xavier for this hidden dim
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module,nn.Embedding):
            torch.nn.init.normal_(module.weight,mean=0.0 , std=0.02)

    def forward(self, idx ,targets=None):
        device=idx.device
        b,t=idx.size() #(B,T)
        assert t <=self.config.block_size, f"sequence length exceeding context limit"
        pos=torch.arange(0,t,dtype=torch.long,device=device)    #(t)

        tok_emb=self.transformer.wte(idx) # ( b,t,nembd)     
        pos_embd=self.transformer.wpe(pos) #(t, n_embd)
        x=self.transformer.drop(tok_emb+pos_embd)
        for block in self.transformer.h:
            x=block(x)
        x=self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits=self.lm_head(x) #(B,T, vocab size)
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)),targets.view(-1),ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits=self.lm_head(x[:,[-1],:])
            loss=None
        return logits,loss

    def crop_block_size(self,block_size):
        #decrease block size if necessary eg we may want to load pretrained gpt-2 which has size 1024 but want 
        #lesser block size for simples smaller model
        assert block_size <= self.config.block_size
        self.config.block_size=block_size
        self.transformer.wpe.weight= nn.Parameter(self.transformer.wpe.weight[:block_size])
        for block in self.transformer.h:
            if hasattr(block.attn, 'bias'):
                block.attn.bias = block.attn.bias[:,:,:block_size,:block_size]

    @classmethod
    def from_pretrained(cls,model_type,override_args=None):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        override_args=override_args or {}
        assert all(k=='dropout' for k in override_args) #only dropout can be overriden
        from transformers import GPT2LMHeadModel
        print(f"loading weights from pretrained gpt {model_type}")
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type] #n_layer, n_head and n_embd are determined from model_type
        print("forcing vocab_size=50257, block_size=1024, bias=True")
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        config_args['bias'] = True # always True for GPT model checkpoints
        if 'dropout' in override_args:
            print(f"overriding dropout rate to {override_args['dropout']}")
            config_args['dropout'] = override_args['dropout']

        #init the from scratch model
        config=GPTConfig(**config_args)
        model=GPT(config)
        sd=model.state_dict()
        sd_keys=sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] #discardinig attention mask/biffer as not a param

        #init hf transformer gpt 
        model_hf=GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf=model_hf.state_dict()
        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them TODO verify
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])
        return model

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        # start with all of the candidate parameters
        param_dict = {pn: p for pn, p in self.named_parameters()}
        # filter out those that do not require grad
        param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
        # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
        
        decay_params = [p for n, p in param_dict.items() if (p.dim() >= 2)]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        num_decay_params = sum(p.numel() for p in decay_params)
        num_nodecay_params = sum(p.numel() for p in nodecay_params)
        print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
        print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
        # Create AdamW optimizer and use the fused version if it is available
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
        print(f"using fused AdamW: {use_fused}")

        return optimizer
    
    def estimate_mfu(self, fwdbwd_per_iter, dt):
        """ estimate model flops utilization (MFU) in units of A100 bfloat16 peak FLOPS """
        # first estimate the number of flops we do per iteration.
        # see PaLM paper Appendix B as ref: https://arxiv.org/abs/2204.02311
        N = self.get_num_params()
        cfg = self.config
        L, H, Q, T = cfg.n_layer, cfg.n_head, cfg.n_embd//cfg.n_head, cfg.block_size
        flops_per_token = 6*N + 12*L*H*Q*T
        flops_per_fwdbwd = flops_per_token * T
        flops_per_iter = flops_per_fwdbwd * fwdbwd_per_iter
        # express our flops throughput as ratio of A100 bfloat16 peak flops
        flops_achieved = flops_per_iter * (1.0/dt) # per second
        flops_promised = 312e12 # A100 GPU bfloat16 peak flops is 312 TFLOPS
        mfu = flops_achieved / flops_promised
        return mfu
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx






In [7]:
# model=GPT.from_pretrained('gpt2')

print('loaded succesfully')

loaded succesfully


In [8]:
# num_return_sequences=5
# max_length= 30

# model.eval()
# model.to(device='cuda')
# torch.manual_seed(42)
# torch.cuda.manual_seed(42)

# import tiktoken
# enc=tiktoken.get_encoding('gpt2')
# tokens=enc.encode("Hello i am a language model,")
# tokens=torch.tensor(tokens,dtype=torch.long) #(B,)
# tokens= tokens.unsqueeze(0).repeat(num_return_sequences,1) #(5,8)
# x=tokens.to('cuda')
# generation=model.generate(x,max_new_tokens=max_length)
# for r in generation.tolist(): # we asked for num return sequence to be 5
#     print(enc.decode(r))


In [9]:
!wget -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-21 10:57:11--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-09-21 10:57:11 (54.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [10]:
if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"

In [ ]:

import tiktoken

#sampling without replacement
class DataLoaderLite:
    def __init__(self, B, T):
        self.B = B
        self.T = T

        # at init load tokens from disk and store them in memory
        with open('input.txt', 'r') as f:
            text = f.read()
        enc = tiktoken.get_encoding('gpt2')
        tokens = enc.encode(text)
        self.tokens = torch.tensor(tokens)
        print(f"loaded {len(self.tokens)} tokens")
        print(f"1 epoch = {len(self.tokens) // (B * T)} batches")

        # state
        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_position : self.current_position+B*T+1] #fetching b times t plus 1
        x = (buf[:-1]).view(B, T) # inputs
        y = (buf[1:]).view(B, T) # targets
        # advance the position in the tensor by B*T
        self.current_position += B * T
        # if loading the next batch would be out of bounds, reset
        if self.current_position + (B * T + 1) > len(self.tokens):
            self.current_position = 0
        return x, y



        

In [37]:
model=GPT(GPTConfig())
model=torch.compile(model)

number of parameters : 123.689472 M


In [38]:
max_steps=50
max_lr=6e-4 #a ccording to gpt3 paper for 124 m model
min_lr=max_lr*0.1 #10 percent of max
warmup_steps=0.2 * max_steps
def get_lr(step):
    #warmup region
    if step < warmup_steps:
        return max_lr * (step+1)/warmup_steps
    #after the decay region
    if step>max_steps:
        return min_lr
    #in between
    decay_ratio= (step - warmup_steps)/(max_steps-warmup_steps)
    assert 0 <= decay_ratio <=1
    coeff=0.5 *(1.0 +math.cos(math.pi *decay_ratio))
    return min_lr + coeff * (max_lr-min_lr)
#gpt also increase the batch size linearly through the training ,skipping it . rational for linearly increasing batch sizes
#is that in early part of the training the gradients are highly colinear , just telling to use  these tokens not to use  others so, teeling these tokens appear these do not appear so the gradients are roughly similiar 
# why to give large batch sizes when the gradients are just highly corelated. later in the trianing gradients become de correlated and thats why larger batch size will help



In [ ]:
#get a data batch
import time


total_batch_size=524288 # 2**19, ~0.5 M in number of tokens
B,T=16,1024 # micro batch size, sequence length
assert total_batch_size %(B*T)== 0, "make sure the total batch size is divisible by zero"
grad_accum_steps= total_batch_size//(B*T)
print(f"total desired batch size {total_batch_size}")
print(f"calculated gradient accumulated steps:{grad_accum_steps}")

train_loader=DataLoaderLite(B=B,T=T)
#get logits
# logits,loss=model(x,y)
model.to(device)

torch.set_float32_matmul_precision('high')

#optimization
# optimizer= torch.optim.AdamW(model.parameters(),lr=3e-4,betas=(0.9,0.95),eps=1e-8,weight_decay=0.1)
optimizer=model.configure_optimizers(weight_decay=0.1,learning_rate=6e-4,device_type=device,betas=(0.9,0.95))

for step in range(max_steps):
    t0=time.time()
    optimizer.zero_grad()
    loss_accum=0.0
    for micro_step in range(grad_accum_steps): #gradient accumulation
        x,y=train_loader.next_batch()
        x,y=x.to(device),y.to(device)
        with torch.autocast(device_type=device,dtype=torch.bfloat16):
            logits,loss=model(x,y)            
        loss=loss/grad_accum_steps
        loss_accum += loss.detach()
        loss.backward()
    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0) #global norm of all parameters
    lr=get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr']=lr
    optimizer.step()
    torch.cuda.synchronize()
    t1=time.time()
    
    dt=(t1-t0) # td in seconds
    dt_ms=dt*1000
    tokens_processed=train_loader.B * train_loader.T * grad_accum_steps
    tokenspersec=(tokens_processed)/(t1-t0)
    print(f'step{step} , loss : {loss_accum.item()},norm: {norm:.4f},lr: {lr:.4e},dt: {dt_ms:.2f}ms , tokens/s: {tokenspersec:.2f}')



total desired batch size 524288
calculated gradient accumulated steps:32
loaded 338025 tokens
1 epoch = 20 batches
num decayed parameter tensors: 50, with 124,354,560 parameters
num non-decayed parameter tensors: 98, with 121,344 parameters
using fused AdamW: True
step0 , loss : 10.981191635131836,norm: 27.1309,lr: 6.0000e-05,dt: 2.72ms , tokens/s: 192820.71
step1 , loss : 9.572906494140625,norm: 8.9545,lr: 1.2000e-04,dt: 2.69ms , tokens/s: 194664.00
step2 , loss : 9.21496868133545,norm: 8.8051,lr: 1.8000e-04,dt: 2.69ms , tokens/s: 194837.46
step3 , loss : 9.706063270568848,norm: 6.6515,lr: 2.4000e-04,dt: 2.70ms , tokens/s: 194412.91
step4 , loss : 9.059660911560059,norm: 4.0322,lr: 3.0000e-04,dt: 2.69ms , tokens/s: 194632.78
step5 , loss : 8.574761390686035,norm: 3.2637,lr: 3.6000e-04,dt: 2.69ms , tokens/s: 194738.11
step6 , loss : 8.265448570251465,norm: 2.3912,lr: 4.2000e-04,dt: 2.69ms , tokens/s: 194554.86
step7 , loss : 7.998306751251221,norm: 2.8046,lr: 4.8000e-04,dt: 2.70ms , to